# A profiled optimum outside its own geometry

This notebook accompanies the docs page
[`ds-geometry-counterexample`](../../docs/examples/ds-geometry-counterexample.md). For the
plain determinant, exchange stability forces the nearest-cell geometry, which is what makes a
terminal partition compilable into a rule. For the profiled criterion it does not — and the
counterexample is not a delicate one. Everything below is exact rational arithmetic, so no
sign depends on a tolerance, and the notebook runs in seconds at any setting of
`SCOREQUANT_EXAMPLE_FAST`.

## Double precision is an application choice

The library never sets global numerical configuration at import time, so the notebook turns
double precision on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

## The fixture

Eight integer score vectors, shifted once by their own exact mean so that the table sums to
zero the way a score sample from a normalized model does under its own reference measure. That
shift is part of constructing a plausible score table, not preprocessing applied to data.
Working in integers keeps every derived quantity a rational with a small denominator.

Bin names carry no meaning, so labelings are enumerated in restricted-growth form: row 0 is in
cell 0, and a new cell may only be opened by the smallest unused index. That visits every
three-cell partition of eight rows exactly once.

In [ ]:
import numpy as np

import scorequant as sq
from examples.ds_geometry_counterexample import (
    RAW_TABLE,
    canonical_labelings,
    efficient_semimetric,
    exact_table,
    make_figure,
    profiled_value,
    run_study,
    violation_margins,
)

table = exact_table()
labelings = canonical_labelings()

print("raw integer table:", list(RAW_TABLE))
print("column sums after the shift:", [sum(row[column] for row in table) for column in (0, 1)])
print("three-cell partitions of eight rows:", len(labelings))

## The exact global optimum

Rank all 966 by the exact profiled value, with score column 0 of interest and column 1 the
nuisance to be estimated from the same labels and profiled away. The winner is clear-cut.

In [ ]:
ranked = sorted(((profiled_value(labels, table), labels) for labels in labelings), reverse=True)
best, optimum = ranked[0]

print("global optimum          ", optimum)
print("its profiled value      ", best, "=", float(best))
print("margin over the runner-up", best - ranked[1][0], "=", float(best - ranked[1][0]))

## The geometry that optimum induces

The gradient of the profiled objective is the inverse binned information minus the embedded
inverse of its nuisance block. Here it has rank one, so its level sets are parallel bands of
constant efficient score: the interest column with the nuisance column regressed out. Ask
where each row belongs under the bands the optimum itself generates.

In [ ]:
margins = violation_margins(optimum, table, efficient_semimetric(optimum, table))

print(f"{'row':>5}{'cell':>6}{'how much farther than the nearest cell':>42}")
print("-" * 53)
for row, margin in enumerate(margins):
    verdict = str(margin) if margin > 0 else "in a nearest cell"
    print(f"{row:>5}{optimum[row]:>6}{verdict:>42}")

Exactly one row is misplaced, by an exact positive rational. Moving it would satisfy the
geometry and would lower the objective, because the labeling it is leaving is the global
optimum of an exhaustive enumeration. There is no tolerance to blame and no local-search
artifact to appeal to.

## The library reproduces it, and refuses to compile it

In [ ]:
from examples.ds_geometry_counterexample import float_table

scores, weights = float_table()
config = sq.DExchangeConfig(seed=1, n_init=32, max_scans=200)
profiled = sq.optimize_partition(
    scores, weights=weights, n_bins=3, criterion=sq.ProfiledDOptimality((0,)), config=config
)
geometry = profiled.profiled_geometry

print("labels from a cold start   ", np.asarray(profiled.labels))
print("profiled information       ", float(np.exp(profiled.objective)), "vs exact", float(best))
print("violating moves            ", geometry.violating_moves)
print("largest violation          ", geometry.maximum_positive_violation)
print("bound residual (must be <=0)", geometry.maximum_bound_residual)
print()
try:
    profiled.compile_quantizer()
except ValueError as error:
    print("compile_quantizer:", error)

## The same eight rows under plain D

Change one argument and the missing implication comes back. The determinant partition of the
same table satisfies its own Mahalanobis rule, compiles into a rule that reproduces its own
labels, and — this being an eight-row problem — can be proved globally optimal outright.

In [ ]:
plain = sq.optimize_partition(scores, weights=weights, n_bins=3, config=config)
compiled = plain.compile_quantizer()
certificate = sq.certify_partition(scores, weights=weights, n_bins=3, incumbent=plain.labels)

print("labels                     ", np.asarray(plain.labels))
print("every row in a nearest cell", plain.geometry.voronoi_consistent)
print("violating moves            ", plain.geometry.violating_moves)
print(
    "compiled rule reproduces them",
    bool(np.array_equal(np.asarray(compiled.predict_scores(scores)), np.asarray(plain.labels))),
)
print("certificate                ", certificate.status, "in", certificate.nodes_explored, "nodes")
print("incumbent was optimal      ", certificate.incumbent_was_optimal)

## Every labeling, ranked

The enumeration answers a sharper question than "does the optimum violate its geometry". It
can ask how many of the 966 labelings satisfy their own, under each criterion. Two of them
have a singular binned information and induce no geometry at all.

In [ ]:
study = run_study()

print(f"{'criterion':<16}{'self-consistent':>18}{'best one ranks':>17}{'value vs optimum':>19}")
print("-" * 70)
for key, name in (("determinant", "plain D"), ("profiled", "profiled D_s")):
    survey = study.metrics[key]
    consistent = f"{len(survey['consistent_ranks'])} of 966"
    print(
        f"{name:<16}{consistent:>18}{survey['best_consistent_rank']:>17}"
        f"{survey['best_consistent_ratio']:>19.6f}"
    )
print()
print("singular labelings, excluded:", study.metrics["profiled"]["singular_labelings"])
print("efficient-score regression coefficient:", study.metrics["efficient_regression"])

Read the profiled row carefully, because it is stronger than the headline. On this table the
efficient-Voronoi geometry does not merely fail to contain the optimum: exactly one labeling
of 966 satisfies it, and that labeling is fifth-best, retaining about 92% of the profiled
information the optimum retains. A solver constrained to produce geometrically self-consistent
profiled labelings would have exactly one choice on this table, and it would be the wrong one.

The plain-D row is the determinant theorem in the form it actually takes. Self-consistency is
necessary at an optimum and not sufficient: five labelings satisfy the Mahalanobis rule, four
of which rank 55th or worse. The theorem says the optimum is among them, and it is.

## The committed figure

In [ ]:
figure = make_figure(study)
figure

## Interpretation

Nothing algebraic breaks for the profiled criterion. Its exchange gain is exact and
closed-form, the solver is monotone, and it terminates — those are statements about a rank-two
update and do not depend on any geometry. What is missing is only the bridge from a stable
labeling to a rule, and this table shows that the bridge is genuinely absent rather than
merely unproved.

That is why `compile_quantizer` refuses on a profiled result instead of handing back a
nearest-cell rule that would disagree with the labels it came from, and why a reusable profiled
rule has to be fitted as one. The theory is
[Chapter 10](../../docs/book/ch10-profiled-ds.md); the fitted-rule path is measured on a real
measurement problem in
[`nuisance-profiled-ds`](../../docs/examples/nuisance-profiled-ds.md).